In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-mpf qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# สูตร Multi-product เพื่อลดความผิดพลาด Trotter

import TutorialFeedback from '@site/src/components/TutorialFeedback';






*ประมาณเวลาใช้งาน QPU: สี่นาทีบนโปรเซสเซอร์ Heron r2 (หมายเหตุ: นี่เป็นเพียงการประมาณเท่านั้น เวลาจริงอาจแตกต่างกัน)*

## พื้นหลัง
บทแนะนำนี้สาธิตวิธีใช้ Multi-Product Formula (MPF) เพื่อให้ได้ความผิดพลาด Trotter ที่ต่ำกว่าใน observable ของเรา เมื่อเทียบกับความผิดพลาดที่เกิดจาก Trotter Circuit ที่ลึกที่สุดที่เราจะรันจริง MPF ลดความผิดพลาด Trotter ของ Hamiltonian dynamics ผ่านการรวมเชิงเส้นแบบถ่วงน้ำหนักจากการรัน Circuit หลายครั้ง พิจารณาการหาค่าคาดหวังของ observable สำหรับสถานะควอนตัม
$\rho(t)=e^{-i H t} \rho(0) e^{i H t}$ ที่มี Hamiltonian $H$ โดยสามารถใช้ Product Formulas (PFs) เพื่อประมาณ time-evolution $e^{-i H t}$ ด้วยวิธีดังนี้:

- เขียน Hamiltonian $H$ ในรูป $H=\sum_{a=1}^d F_a,$ โดยที่ $F_a$ เป็น Hermitian operators ซึ่งสามารถนำ unitary ที่สอดคล้องกันมาใช้งานบนอุปกรณ์ควอนตัมได้อย่างมีประสิทธิภาพ
- ประมาณค่าพจน์ $F_a$ ที่ไม่ commute ซึ่งกันและกัน

จากนั้น first-order PF (สูตร Lie-Trotter) คือ:

$$S_1(t):=\prod_{a=1}^d e^{-i F_a t},$$

ซึ่งมี error term เป็น quadratic $S_1(t)=e^{-i H t}+\mathcal{O}\left(t^{2}\right)$ นอกจากนี้ยังสามารถใช้ higher-order PFs (สูตร Lie-Trotter-Suzuki) ซึ่งมีการลู่เข้าที่เร็วกว่าและนิยามแบบ recursive ดังนี้:

$$S_2(t):=\prod_{a=1}^d e^{-i F_a t/2}\prod_{a=1}^d e^{-i F_a t/2}$$

$$S_{2 \chi}(t):= S_{2 \chi -2}(s_{\chi}t)^2 S_{2 \chi -2}((1-4s_{\chi})t)S_{2 \chi -2}(s_{\chi}t)^2,$$

โดยที่ $\chi$ คือลำดับของ symmetric PF และ $s_p = \left( 4 - 4^{1/(2p-1)} \right)^{-1}$ สำหรับ time-evolution ระยะยาว สามารถแบ่งช่วงเวลา $t$ ออกเป็น $k$ ช่วง เรียกว่า Trotter steps มีขนาด $t/k$ แต่ละช่วง และประมาณ time-evolution ในแต่ละช่วงด้วย product formula อันดับ $\chi$ คือ $S_{\chi}$ ดังนั้น PF อันดับ $\chi$ สำหรับ time-evolution operator ใน $k$ Trotter steps คือ:

$$ S_{\chi}^{k}(t) = \left[ S_{\chi} \left( \frac{t}{k} \right)\right]^k = e^{-i H t}+O\left(t \left( \frac{t}{k} \right)^{\chi} \right)$$

โดย error term จะลดลงตามจำนวน Trotter steps $k$ และลำดับ $\chi$ ของ PF

สำหรับจำนวนเต็ม $k \geq 1$ และ product formula $S_{\chi}(t)$ สถานะที่ถูก time-evolve โดยประมาณ $\rho_k(t)$ สามารถหาได้จาก $\rho_0$ โดยการใช้ product formula $S_{\chi}\left(\frac{t}{k}\right)$ ซ้ำ $k$ ครั้ง

$$
\rho_k(t)=S_{\chi}\left(\frac{t}{k}\right)^k \rho_0 S_{\chi}\left(\frac{t}{k}\right)^{-k}
$$

$\rho_k(t)$ เป็นการประมาณของ $\rho(t)$ โดยมี Trotter approximation error คือ ||$\rho_k(t)-\rho(t) ||$ ถ้าพิจารณา linear combination ของ Trotter approximations ของ $\rho(t)$:

$$
\mu(t) = \sum_{j}^{l} x_j \rho^{k_j}_{j}\left(\frac{t}{k_j}\right) + \text{some remaining Trotter error} \, ,
$$

โดยที่ $x_j$ คือ weighting coefficients ของเรา $\rho^{k_j}_j$ คือ density matrix ที่สอดคล้องกับ pure state ที่ได้จากการ evolve สถานะเริ่มต้นด้วย product formula $S^{k_j}_{\chi}$ ที่ใช้ $k_j$ Trotter steps และ $j \in {1, ..., l}$ เป็น index ของจำนวน PFs ที่ประกอบขึ้นเป็น MPF ทุกพจน์ใน $\mu(t)$ ใช้ product formula $S_{\chi}(t)$ เดียวกันเป็นฐาน
เป้าหมายคือการปรับปรุง ||$\rho_k(t)-\rho(t) \|$ โดยหา $\mu(t)$ ที่มี $\|\mu(t)-\rho(t)\|$ ต่ำกว่า

* $\mu(t)$ ไม่จำเป็นต้องเป็นสถานะทางกายภาพ เนื่องจาก $x_i$ ไม่จำเป็นต้องเป็นบวก เป้าหมายที่นี่คือการลดความผิดพลาดในค่าคาดหวังของ observables ไม่ใช่การหาตัวแทนทางกายภาพของ $\rho(t)$
* $k_j$ กำหนดทั้งความลึกของ Circuit และระดับของ Trotter approximation ค่า $k_j$ ที่เล็กกว่าจะให้ Circuit ที่สั้นกว่าซึ่งมีความผิดพลาดของ Circuit น้อยกว่า แต่จะเป็นการประมาณสถานะที่ต้องการได้แม่นยำน้อยกว่า

ประเด็นสำคัญคือ Trotter error ที่เหลืออยู่ที่ $\mu(t)$ ให้มานั้นเล็กกว่า Trotter error ที่จะได้จากการใช้ค่า $k_j$ ที่ใหญ่ที่สุดเพียงอย่างเดียว

สามารถมองประโยชน์ของสิ่งนี้ได้จากสองมุมมอง:

1. สำหรับงบประมาณ Trotter steps ที่กำหนดไว้ที่สามารถรันได้ สามารถได้ผลลัพธ์ที่มี Trotter error ต่ำกว่าโดยรวม
2. สำหรับจำนวน Trotter steps เป้าหมายที่มากเกินไปจะรัน สามารถใช้ MPF เพื่อหาชุด Circuit ความลึกต่ำกว่าที่จะรัน ซึ่งให้ Trotter error ที่ใกล้เคียงกัน

## ข้อกำหนด
ก่อนเริ่มบทแนะนำนี้ ตรวจสอบว่าติดตั้งสิ่งต่อไปนี้แล้ว:

* Qiskit SDK v1.0 ขึ้นไป พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.22 ขึ้นไป (`pip install qiskit-ibm-runtime`)
* MPF Qiskit addons (`pip install qiskit_addon_mpf`)
* Qiskit addons utils (`pip install qiskit_addon_utils`)
* ไลบรารี Quimb (`pip install quimb`)
* ไลบรารี Qiskit Quimb (`pip install qiskit-quimb`)
* Numpy v0.21 สำหรับความเข้ากันได้ข้ามแพ็กเกจ (`pip install numpy==0.21`)

## ส่วนที่ I ตัวอย่างขนาดเล็ก
### สำรวจความเสถียรของ MPF
ไม่มีข้อจำกัดที่ชัดเจนเกี่ยวกับการเลือกจำนวน Trotter steps $k_j$ ที่ประกอบขึ้นเป็นสถานะ MPF $\mu(t)$ อย่างไรก็ตาม ต้องเลือกอย่างระมัดระวังเพื่อหลีกเลี่ยงความไม่เสถียรในค่าคาดหวังที่คำนวณจาก $\mu(t)$ กฎทั่วไปที่ดีคือตั้งค่า Trotter step ที่เล็กที่สุด $k_{\text{min}}$ ให้ $t/k_{\text{min}} \lt 1$ ถ้าต้องการเรียนรู้เพิ่มเติมเกี่ยวกับเรื่องนี้และวิธีเลือกค่า $k_j$ อื่น ๆ ดูได้ที่คู่มือ [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html)

ในตัวอย่างด้านล่างนี้ เราสำรวจความเสถียรของ MPF solution โดยคำนวณค่าคาดหวังของ magnetization สำหรับช่วงเวลาต่าง ๆ โดยใช้สถานะ time-evolved ที่แตกต่างกัน โดยเฉพาะอย่างยิ่ง เราเปรียบเทียบค่าคาดหวังที่คำนวณจาก time-evolution โดยประมาณแต่ละแบบที่ใช้ Trotter steps ที่สอดคล้องกัน และโมเดล MPF ต่าง ๆ (static และ dynamic coefficients) กับค่าที่แม่นยำของ observable ที่ถูก time-evolve ก่อนอื่น มากำหนดพารามิเตอร์สำหรับสูตร Trotter และช่วงเวลาที่ต้องการ

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from copy import deepcopy

from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import XXPlusYYGate
from qiskit.transpiler.passes.optimization.collect_and_collapse import (
    CollectAndCollapse,
    collect_using_filter_function,
    collapse_to_operation,
)

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit_addon_utils.problem_generators import (
    generate_xyz_hamiltonian,
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth
from qiskit_addon_mpf.static import setup_static_lse
from qiskit_addon_mpf.dynamic import setup_dynamic_lse
from qiskit_addon_mpf.costs import (
    setup_exact_problem,
    setup_sum_of_squares_problem,
    setup_frobenius_problem,
)
from qiskit_addon_mpf.backends.tenpy_layers import (
    LayerModel,
    LayerwiseEvolver,
)
from qiskit_addon_mpf.backends.tenpy_tebd import MPOState, MPS_neel_state

from scipy.linalg import expm

# Suppress TeNPy's `unit_cell_width` future-API warning. The default
# (`unit_cell_width=len(sites)`) is correct for Chain lattices, which is what
# `CouplingMap.from_line(...)` produces here, so the warning is informational.
warnings.filterwarnings(
    "ignore",
    message=r".*unit_cell_width.*",
    category=UserWarning,
)


# --- Helper: collect XX + YY rotations into a single gate ---
def filter_function(node):
    return node.op.name in {"rxx", "ryy"}


collect_function = partial(
    collect_using_filter_function,
    filter_function=filter_function,
    split_blocks=True,
    min_block_size=1,
)


def collapse_to_xx_plus_yy(block):
    param = 0.0
    for node in block.data:
        param += node.operation.params[0]
    return XXPlusYYGate(param)


collapse_function = partial(
    collapse_to_operation,
    collapse_function=collapse_to_xx_plus_yy,
)

pm = PassManager()
pm.append(CollectAndCollapse(collect_function, collapse_function))

สำหรับตัวอย่างนี้เราจะใช้สถานะ Neel เป็นสถานะเริ่มต้น $\vert \text{Neel} \rangle = \vert 0101...01 \rangle$ และโมเดล Heisenberg บนเส้นตรง 10 sites สำหรับ Hamiltonian ที่ควบคุม time-evolution

$$
\hat{\mathcal{H}}_{Heis} = J \sum_{i=1}^{L-1} \left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ Z_i Z_{(i+1)} \right) \, ,
$$

โดยที่ $J$ คือความแข็งแรงของการจับคู่สำหรับเส้นเชื่อมระหว่างเพื่อนบ้านที่ใกล้ที่สุด

In [2]:
L = 10

# Generate coupling map and Hamiltonian
coupling_map = CouplingMap.from_line(L, bidirectional=False)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(1.0, 1.0, 1.0),
    ext_magnetic_field=(0.0, 0.0, 0.0),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


In [3]:
# Observable: ZZ on the middle pair of qubits
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)
print(observable)

SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


In [4]:
# MPF parameters
mpf_trotter_steps = [1, 2, 4]
order = 2
symmetric = False

trotter_times = np.arange(0.5, 1.55, 0.1)
exact_evolution_times = np.arange(trotter_times[0], 1.55, 0.05)

#### Build Trotter circuits

We create the circuits implementing the approximate Trotter time-evolutions for each time point and each Trotter step count. The `CollectAndCollapse` pass defined in the Setup section collects XX and YY rotations into single XX+YY gates, to prepare for more efficient tensor-network simulation later.

In [5]:
# Initial Neel state preparation
initial_state_circ = QuantumCircuit(L)
initial_state_circ.x([i for i in range(L) if i % 2 != 0])


all_circs = []
for total_time in trotter_times:
    mpf_trotter_circs = [
        generate_time_evolution_circuit(
            hamiltonian,
            time=total_time,
            synthesis=SuzukiTrotter(reps=num_steps, order=order),
        )
        for num_steps in mpf_trotter_steps
    ]

    mpf_trotter_circs = pm.run(
        mpf_trotter_circs
    )  # Collect XX and YY into XX + YY

    mpf_circuits = [
        initial_state_circ.compose(circuit) for circuit in mpf_trotter_circs
    ]
    all_circs.append(mpf_circuits)

In [6]:
mpf_circuits[-1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/c7ee61e7-0.avif" alt="Output of the previous code cell" />

จากนั้นเราสร้าง Circuit ที่ใช้งาน Trotter time-evolution โดยประมาณ

In [7]:
aer_sim = AerSimulator()
pm_sim = generate_preset_pass_manager(backend=aer_sim, optimization_level=3)

isa_circs_all_times = [
    pm_sim.run([deepcopy(c) for c in mpf_circuits])
    for mpf_circuits in all_circs
]

### Step 3: Execute using Qiskit primitives

For the small-scale example we run the ISA-lowered Trotter circuits through the `EstimatorV2` primitive backed by Aer. Doing so gives us a *noiseless* reference value for each $(k_j, t)$ pair &mdash; these are the $\langle A \rangle_{k_j}(t)$ values that the MPF will combine in Step 4. We sweep over evolution times so that we can later plot the full time-series curve of each individual product formula and of the MPF.

In [8]:
estimator = Estimator(mode=aer_sim)

mpf_expvals_all_times, mpf_stds_all_times = [], []
for isa_circuits in isa_circs_all_times:
    result = estimator.run(
        [(circuit, observable) for circuit in isa_circuits], precision=0.005
    ).result()
    mpf_expvals_all_times.append([res.data.evs for res in result])
    mpf_stds_all_times.append([res.data.stds for res in result])

![Output of the previous code cell](../docs/images/tutorials/multi-product-formula/extracted-outputs/92dc20a7-0.avif)

ต่อไป เราคำนวณค่าคาดหวัง time-evolved จาก Trotter Circuit

In [9]:
exact_expvals = []
for t in exact_evolution_times:
    exp_H = expm(-1j * t * hamiltonian.to_matrix())
    initial_state = Statevector(initial_state_circ).data
    time_evolved_state = exp_H @ initial_state

    exact_obs = (
        time_evolved_state.conj()
        @ observable.to_matrix()
        @ time_evolved_state
    ).real
    exact_expvals.append(exact_obs)

เรายังคำนวณค่าคาดหวังที่แม่นยำสำหรับการเปรียบเทียบด้วย

In [10]:
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)

#### Static MPF coefficients
Static MPF คือ MPF ที่ค่า $x_j$ ไม่ขึ้นกับเวลา evolution $t$ ลองพิจารณา PF อันดับ $\chi = 1$ ที่มี $k_j$ Trotter steps สามารถเขียนได้เป็น:

$$ S_1^{k_j}\left( \frac{t}{k_j} \right)=e^{-i H t}+ \sum_{n=1}^{\infty} A_n \frac{t^{n+1}}{k_j^n}  $$

โดยที่ $A_n$ คือเมทริกซ์ที่ขึ้นกับ commutators ของพจน์ $F_a$ ในการ decompose ของ Hamiltonian สิ่งสำคัญคือต้องสังเกตว่า $A_n$ เองนั้นเป็นอิสระจากเวลาและจำนวน Trotter steps $k_j$ ดังนั้นจึงเป็นไปได้ที่จะตัด lower-order error terms ที่มีส่วนร่วมใน $\mu(t)$ ด้วยการเลือก weights $x_j$ ของ linear combination อย่างระมัดระวัง เพื่อตัด Trotter error สำหรับ $l-1$ พจน์แรก (พจน์เหล่านี้จะมีส่วนร่วมมากที่สุดเนื่องจากสอดคล้องกับจำนวน Trotter steps ที่น้อยกว่า) ในนิพจน์ของ $\mu(t)$ coefficients $x_j$ ต้องเป็นไปตามสมการดังนี้:

$$ \sum_{j=1}^l x_j = 1 $$
$$ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{n}} = 0 $$

โดย $n=0, ... l-2$ สมการแรกรับประกันว่าไม่มี bias ในสถานะ $\mu(t)$ ที่สร้างขึ้น ในขณะที่สมการที่สองรับประกันการตัด Trotter errors สำหรับ PF อันดับสูงกว่า สมการที่สองกลายเป็น $ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{\eta}} = 0 $ โดยที่ $\eta = \chi + 2n$ สำหรับ symmetric PFs และ $\eta = \chi + n$ สำหรับกรณีอื่น ๆ โดย $n=0, ..., l-2$ ความผิดพลาดที่ได้ (อ้างอิง [\[1\]](#references),[\[2\]](#references)) คือ

$$ \epsilon = \mathcal{O} \left( \frac{t^{l+1}}{k_1^l} \right).$$

การหา static MPF coefficients สำหรับชุดค่า $k_j$ ที่กำหนดนั้นเทียบเท่ากับการแก้ระบบสมการเชิงเส้นที่นิยามโดยสมการทั้งสองข้างต้นสำหรับตัวแปร $x_j$: $Ax=b$ โดยที่ $x$ คือ coefficients ที่เราต้องการ $A$ คือเมทริกซ์ที่ขึ้นกับ $k_j$ และประเภทของ PF ที่ใช้ ($S$) และ $b$ คือเวกเตอร์ของ constraints โดยเฉพาะอย่างยิ่ง:

$$A_{0,j} = 1 $$
$$A_{i>0,j} = k_{j}^{-(\chi + s(i-1))}$$
$$b_0 = 1$$
$$b_{i>0} = 0 $$

โดยที่ $\chi$ คือ ``order``, $s$ คือ $2$ ถ้า ``symmetric`` เป็น ``True`` และ $1$ สำหรับกรณีอื่น ๆ $k_{j}$ คือ ``trotter_steps`` และ $x$ คือตัวแปรที่ต้องการแก้ index $i$ และ $j$ เริ่มที่ $0$ เราสามารถแสดงในรูปเมทริกซ์ได้เช่นกัน:

$$
A =
\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} & ... \\
A_{1,0} & A_{1,1} & A_{1,2} & ...  \\
A_{2,0} & A_{2,1} & A_{2,2} & ...  \\
... & ... & ... & ...
\end{bmatrix} =
\begin{bmatrix}
1 & 1 & 1 & ... \\
k_{0}^{-(\chi + s(1-1))} & k_{1}^{-(\chi + s(1-1))} & k_{2}^{-(\chi + s(1-1))} & ... \\
k_{0}^{-(\chi + s(2-1))} & k_{1}^{-(\chi + s(2-1))} & k_{2}^{-(\chi + s(2-1))} & ... \\
... & ... & ... & ...
\end{bmatrix}
$$

และ

$$
b =
\begin{bmatrix}
b_{0} \\
b_{1} \\
b_{2}  \\
...
\end{bmatrix} =
\begin{bmatrix}
1 \\
0 \\
0  \\
...
\end{bmatrix}
$$

สำหรับรายละเอียดเพิ่มเติม ดูเอกสารของ Linear System of Equations ([LSE](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.LSE.html))

เราสามารถหาคำตอบสำหรับ $x$ ได้เชิงวิเคราะห์เป็น $x = A^{-1}b$ ดูตัวอย่างในอ้างอิง [\[1\]](#references) หรือ [\[2\]](#references) อย่างไรก็ตาม คำตอบที่แม่นยำนี้อาจ "ill-conditioned" ส่งผลให้ L1-norm ของ coefficients $x$ ของเราใหญ่มาก ซึ่งอาจทำให้ MPF มีประสิทธิภาพต่ำ แต่สามารถใช้คำตอบโดยประมาณที่ minimize L1-norm ของ $x$ แทน เพื่อพยายาม optimize พฤติกรรม MPF

##### ตั้งค่า LSE
เมื่อเลือกค่า $k_j$ แล้ว ขั้นแรกต้องสร้าง LSE ก่อน คือ $Ax=b$ ตามที่อธิบายข้างต้น เมทริกซ์ $A$ ขึ้นกับไม่เพียงแค่ $k_j$ แต่ยังขึ้นกับตัวเลือก PF ของเรา โดยเฉพาะ _order_ ของมัน นอกจากนี้อาจต้องพิจารณาว่า PF เป็น symmetric หรือไม่ (ดู [\[1\]](#references)) โดยตั้งค่า `symmetric=True/False` อย่างไรก็ตามนี่ไม่ใช่ข้อบังคับ ตามที่อ้างอิง [\[2\]](#references) แสดงให้เห็น

In [11]:
lse.A

array([[1.      , 1.      , 1.      ],
       [1.      , 0.25    , 0.0625  ],
       [1.      , 0.125   , 0.015625]])

In [12]:
lse.b

array([1., 0., 0.])

With the LSE in hand, we solve for the static coefficients $x_j$ via `lse.solve()` (this is the direct $x = A^{-1}b$ solution).

In [13]:
mpf_coeffs = lse.solve()
print(
    f"The static coefficients associated with the ansatze are: {mpf_coeffs}"
)

The static coefficients associated with the ansatze are: [ 0.04761905 -0.57142857  1.52380952]


ส่วนเวกเตอร์ของ constraints $b$ มี elements ดังนี้:
$$ b_{0} = 1 $$
$$ b_1 = b_2 = 0 $$

ดังนั้น

$$
b =
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

และในทำนองเดียวกันใน `lse`:

In [14]:
model_exact, coeffs_exact = setup_exact_problem(lse)
model_exact.solve()
print(coeffs_exact.value)

[ 0.04761905 -0.57142857  1.52380952]


In [15]:
print(
    "L1 norm of the exact coefficients:",
    np.linalg.norm(coeffs_exact.value, ord=1),
)

L1 norm of the exact coefficients: 2.1428571428556378


Object `lse` มีเมธอดสำหรับหา static coefficients $x_j$ ที่เป็นไปตามระบบสมการ

In [16]:
model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=1.5
)
model_approx.solve()
print(coeffs_approx.value)
print(
    "L1 norm of the approximate coefficients:",
    np.linalg.norm(coeffs_approx.value, ord=1),
)

[-1.10294118e-03 -2.48897059e-01  1.25000000e+00]
L1 norm of the approximate coefficients: 1.5


#### Dynamic MPF coefficients

The static MPF cancels Trotter-error terms in a Hamiltonian- and state-agnostic way, so it does not necessarily produce the smallest possible approximation error for a given Hamiltonian and initial state. The dynamic MPF (Refs. [\[2\]](#references), [\[3\]](#references)) instead finds time-dependent coefficients $x_i(t)$ that minimize the Frobenius-norm distance $\|\rho(t) - \mu^D(t)\|_F^2$ at each time $t$. As shown in the Background, this requires the overlap matrix $M_{ij}(t)$ between Trotter-evolved states and the overlap $L_i(t)$ with the exact state &mdash; both of which we estimate using tensor-network (TeNPy) backends in `qiskit_addon_mpf`.

To set up the dynamic LSE we need three ingredients:

1. An **approximate evolver factory** that the addon will run for each $k_j$ to produce $\rho_{k_j}(t)$ as an MPS/MPO. We build it from the layered structure of the order-$2$ Trotter circuit (one layer per `slice_by_depth`), wrapped as a `LayerwiseEvolver` with TeNPy truncation parameters.
2. An **exact evolver factory** that produces a high-accuracy reference $\rho(t)$. We use a small-time-step fourth-order Suzuki-Trotter circuit (`dt=0.1`, `order=4`) as a proxy for exact evolution.
3. An **identity factory** and an **initial-state MPS** that seed the TeNPy simulation.

The cell below constructs the approximate evolver factory.

In [17]:
# Create approximate time-evolution circuits
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)  # collect XX and YY

# Find layers in the circuit
layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)

# Create tensor network models
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

# Create the time-evolution object
approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

##### หาค่า $x$ ที่เหมาะสมโดยใช้ exact model
นอกจากการคำนวณ $x=A^{-1}b$ แล้ว ยังสามารถใช้ [setup_exact_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_exact_model.html) เพื่อสร้าง instance ของ [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) ที่ใช้ LSE เป็น constraints และ optimal solution ของมันจะให้ค่า $x$

ในส่วนถัดไปจะเห็นได้ชัดว่าทำไม interface นี้จึงมีอยู่

In [18]:
single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

Finally, we define an `identity_factory` that yields the initial MPO state and prepare the Néel initial state as an MPS that matches the lattice used by the layered Trotter model.

In [19]:
def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

เป็นตัวบ่งชี้ว่า MPF ที่สร้างด้วย coefficients เหล่านี้จะให้ผลลัพธ์ที่ดีหรือไม่ เราสามารถใช้ L1-norm (ดูอ้างอิง [\[1\]](#references) ด้วย)

In [20]:
mpf_dynamic_coeffs_list = []
for t in trotter_times:
    print(f"Computing dynamic coefficients for time={t}")
    lse = setup_dynamic_lse(
        mpf_trotter_steps,
        t,
        identity_factory,
        exact_factory,
        approx_factory,
        mps_initial_state,
    )
    problem, coeffs = setup_frobenius_problem(lse)
    try:
        problem.solve()
        mpf_dynamic_coeffs_list.append(coeffs.value)
    except Exception as error:
        mpf_dynamic_coeffs_list.append(np.zeros(len(mpf_trotter_steps)))
        print(error, "Calculation Failed for time", t)
    print("")

Computing dynamic coefficients for time=0.5

Computing dynamic coefficients for time=0.6

Computing dynamic coefficients for time=0.7

Computing dynamic coefficients for time=0.7999999999999999

Computing dynamic coefficients for time=0.8999999999999999

Computing dynamic coefficients for time=0.9999999999999999

Computing dynamic coefficients for time=1.0999999999999999

Computing dynamic coefficients for time=1.1999999999999997

Computing dynamic coefficients for time=1.2999999999999998

Computing dynamic coefficients for time=1.4

Computing dynamic coefficients for time=1.4999999999999998



#### Combine Trotter expectation values with the MPF coefficients

Now we evaluate $\langle A \rangle_{\text{MPF}}(t) = \sum_j x_j \, \langle A \rangle_{k_j}(t)$ for each set of coefficients (static-exact, static-approximate, and dynamic), propagate the per-circuit standard errors, and plot the resulting time series against the exact-diagonalization curve.

In [21]:
sym = {1: "^", 2: "s", 4: "p"}
# Get expectation values at all times for each Trotter step
for k, step in enumerate(mpf_trotter_steps):
    trotter_curve, trotter_curve_error = [], []
    for trotter_expvals, trotter_stds in zip(
        mpf_expvals_all_times, mpf_stds_all_times
    ):
        trotter_curve.append(trotter_expvals[k])
        trotter_curve_error.append(trotter_stds[k])

    plt.errorbar(
        trotter_times,
        trotter_curve,
        yerr=trotter_curve_error,
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

# Get expectation values at all times for the static MPF with exact coeffs
exact_mpf_curve, exact_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_exact.value, trotter_stds)
            ]
        )
    )
    exact_mpf_curve_error.append(mpf_std)
    exact_mpf_curve.append(trotter_expvals @ coeffs_exact.value)

plt.errorbar(
    trotter_times,
    exact_mpf_curve,
    yerr=exact_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Exact",
    color="purple",
)


# Get expectation values at all times for the static MPF with approximate coeffs
approx_mpf_curve, approx_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_approx.value, trotter_stds)
            ]
        )
    )
    approx_mpf_curve_error.append(mpf_std)
    approx_mpf_curve.append(trotter_expvals @ coeffs_approx.value)

plt.errorbar(
    trotter_times,
    approx_mpf_curve,
    yerr=approx_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Approx",
    color="orange",
)


# Get expectation values at all times for the dynamic MPF
dynamic_mpf_curve, dynamic_mpf_curve_error = [], []
for trotter_expvals, trotter_stds, dynamic_coeffs in zip(
    mpf_expvals_all_times, mpf_stds_all_times, mpf_dynamic_coeffs_list
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(dynamic_coeffs, trotter_stds)
            ]
        )
    )
    dynamic_mpf_curve_error.append(mpf_std)
    dynamic_mpf_curve.append(trotter_expvals @ dynamic_coeffs)

plt.errorbar(
    trotter_times,
    dynamic_mpf_curve,
    yerr=dynamic_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Dynamic MPF",
    color="pink",
)


# Exact expectation values
plt.plot(
    exact_evolution_times,
    exact_expvals,
    color="red",
    linestyle="--",
    label="Exact time-evolution",
)

plt.title(f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ vs time")
plt.xlabel("Time")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/35042576-0.avif" alt="Output of the previous code cell" />

##### หาค่า $x$ ที่เหมาะสมโดยใช้ approximate model
อาจเกิดขึ้นได้ที่ L1 norm สำหรับชุดค่า $k_j$ ที่เลือกถูกพิจารณาว่าสูงเกินไป หากเป็นเช่นนั้นและไม่สามารถเลือกชุดค่า $k_j$ ที่ต่างออกไปได้ สามารถใช้คำตอบโดยประมาณของ LSE แทนคำตอบที่แม่นยำ

ทำได้โดยใช้ [setup_approximate_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_approximate_model.html) เพื่อสร้าง instance ของ [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) ที่แตกต่างกัน ซึ่งจำกัด L1-norm ไว้ที่ threshold ที่เลือก โดย minimize ความแตกต่างของ $Ax$ และ $b$

In [22]:
# -------------------------Step 1-------------------------
L = 50
coupling_map = CouplingMap.from_line(L, bidirectional=False)

# XXZ Hamiltonian with random couplings (Ref. [3])
np.random.seed(0)
even_edges = list(coupling_map.get_edges())[::2]
odd_edges = list(coupling_map.get_edges())[1::2]

Js = np.random.uniform(0.5, 1.5, size=L)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), 2 * Js[i]),
            ("YY", (edge), 2 * Js[i]),
            ("ZZ", (edge), 4 * Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

total_time = 3
mpf_trotter_steps = [3, 4, 6]
order = 2
symmetric = True

# Static coefficients
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)
mpf_coeffs = lse.solve()
print(f"Static coefficients: {mpf_coeffs}")
print(f"L1 norm: {np.linalg.norm(mpf_coeffs, ord=1)}")

model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=2.0
)
model_approx.solve()
print(f"Approximate coefficients: {coeffs_approx.value}")
print(f"L1 norm (approx): {np.linalg.norm(coeffs_approx.value, ord=1)}")

# -------------------------Dynamic coefficients-------------------------
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)

layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 4,
    },
)

single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 3,
    },
)


def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

print(f"Computing dynamic coefficients for time={total_time}")
lse_dyn = setup_dynamic_lse(
    mpf_trotter_steps,
    total_time,
    identity_factory,
    exact_factory,
    approx_factory,
    mps_initial_state,
)
problem, coeffs_dyn = setup_frobenius_problem(lse_dyn)
try:
    problem.solve()
    mpf_dynamic_coeffs = coeffs_dyn.value
except Exception as error:
    mpf_dynamic_coeffs = np.zeros(len(mpf_trotter_steps))
    print(error, "Calculation Failed")

# -------------------------Step 1 (cont): Build circuits-------------------------
mpf_circuits = []
for k in mpf_trotter_steps:
    circuit = QuantumCircuit(L)
    circuit.x([i for i in range(L) if i % 2])
    trotter_circ = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=k, order=order),
        time=total_time,
    )
    circuit.compose(trotter_circ, qubits=range(L), inplace=True)
    mpf_circuits.append(circuit)

# Baseline "single deep circuit" comparison run with k=10 Trotter steps.
# Its two-qubit depth is deeper than the deepest MPF constituent (k_max=6) plus
# the overhead of running multiple circuits, pushing it into the noise-limited
# regime where MPF is expected to outperform. It does NOT target the MPF's effective
# Trotter error (which would require many more steps).
comp_circuit = QuantumCircuit(L)
comp_circuit.x([i for i in range(L) if i % 2])
trotter_circ = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=10, order=order),
    time=total_time,
)
comp_circuit.compose(trotter_circ, qubits=range(L), inplace=True)
mpf_circuits.append(comp_circuit)

Static coefficients: [ 0.42857143 -1.82857143  2.4       ]
L1 norm: 4.65714285714286
Approximate coefficients: [-0.4942491   0.40206845  1.09218065]
L1 norm (approx): 1.9884981979026675
Computing dynamic coefficients for time=3


We now optimize the circuits for the chosen backend. We use the Qiskit preset pass manager at `optimization_level=3`, which automatically selects a good set of physical qubits and routes each circuit onto the device topology.

In [23]:
# -------------------------Step 2-------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=L)
backend = service.backend("ibm_fez")
print(backend)

transpiler = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)
transpiled_circuits = [transpiler.run(circ) for circ in mpf_circuits]

isa_observables = [
    observable.apply_layout(circ.layout) for circ in transpiled_circuits
]

<IBMBackend('ibm_fez')>


สังเกตว่ามีอิสระสมบูรณ์ในการแก้ปัญหา optimization นี้ ซึ่งหมายความว่าสามารถเปลี่ยน optimization solver, convergence thresholds ของมัน และอื่น ๆ ได้ ดูคู่มือที่เกี่ยวข้องเรื่อง [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html).

#### Dynamic MPF coefficients
ในส่วนก่อนหน้า เราแนะนำ static MPF ที่ปรับปรุงจาก Trotter approximation มาตรฐาน อย่างไรก็ตาม static version นี้ไม่จำเป็นต้อง minimize approximation error เสมอไป โดยเฉพาะอย่างยิ่ง static MPF ที่แทนด้วย $\mu^S(t)$ ไม่ใช่ optimal projection ของ $\rho(t)$ ลงบน subspace ที่สร้างขึ้นโดยสถานะ product-formula ${\rho_{k_i}(t)}_{i=1}^r$

เพื่อแก้ไขปัญหานี้ เราพิจารณา dynamic MPF (แนะนำในอ้างอิง [\[2\]](#references) และแสดงให้เห็นเชิงทดลองในอ้างอิง [\[3\]](#references)) ที่ minimize approximation error ใน Frobenius norm อย่างแท้จริง โดยเป็นทางการ เราเน้นที่การ minimize

$$
\|\rho(t) - \mu^D(t)\|_F^2 \;=\; \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr],
$$

สำหรับ coefficients $x_i(t)$ บางตัวในแต่ละเวลา $t$ *optimal* projector ใน Frobenius norm คือ $\mu^D(t) \;=\; \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t)$ และเราเรียก $\mu^D(t)$ ว่า *dynamic* MPF เมื่อแทนค่านิยามข้างต้น:

$$
\|\rho(t) - \mu^D(t)\|_F^2
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr]
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t) \right) \left(  \rho(t) - \sum_{j=1}^r x_j(t)\,\rho_{k_j}(t) \right) \bigr]
\;=\; \\
= 1 \;+\; \sum_{i,j=1}^r M_{i,j}(t)\,x_i(t)\,x_j(t)
\;-\;
2 \sum_{i=1}^r L_i^{\mathrm{exact}}(t)\,x_i(t),
$$

โดยที่ $M_{i,j}(t)$ คือ *Gram matrix* นิยามโดย

$$
M_{i,j}(t) \;=\; \mathrm{Tr}\bigl[\rho_{k_i}(t)\,\rho_{k_j}(t)\bigr]
\;=\;
\bigl|\langle \psi_{\mathrm{in}} \!\mid S\bigl(t/k_i\bigr)^{-k_i}\,S\bigl(t/k_j\bigr)^{k_j} \!\mid \psi_{\mathrm{in}} \rangle \bigr|^2.
$$

และ

$$
L_i^{\mathrm{exact}}(t) = \mathrm{Tr}[\rho(t)\,\rho_{k_i}(t)]
$$

แทน overlap ระหว่างสถานะที่แม่นยำ $\rho(t)$ และ product-formula approximation $\rho_{k_i}(t)$ แต่ละตัว ในสถานการณ์จริง overlaps เหล่านี้อาจวัดได้เพียงโดยประมาณเนื่องจาก noise หรือการเข้าถึง $\rho(t)$ บางส่วน

ที่นี่ $\lvert\psi_{\mathrm{in}}\rangle$ คือสถานะเริ่มต้น และ $S(\cdot)$ คือการดำเนินการที่ใช้ใน product formula การเลือก coefficients $x_i(t)$ ที่ minimize นิพจน์นี้ (และจัดการกับข้อมูล overlap โดยประมาณเมื่อ $\rho(t)$ ไม่เป็นที่ทราบอย่างสมบูรณ์) ทำให้ได้ dynamic approximation ที่ "ดีที่สุด" (ในแง่ Frobenius-norm) ของ $\rho(t)$ ภายใน MPF subspace ปริมาณ $L_i(t)$ และ $M_{i,j}(t)$ สามารถคำนวณได้อย่างมีประสิทธิภาพโดยใช้ tensor network methods [\[3\]](#references) MPF Qiskit addon มี "backends" หลายตัวสำหรับการคำนวณนี้ ตัวอย่างด้านล่างแสดงวิธีที่ยืดหยุ่นที่สุด และเอกสาร [TeNPy layer-based backend](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.tenpy_layers.html#module-qiskit_addon_mpf.backends.tenpy_layers) ยังอธิบายรายละเอียดอย่างครบถ้วน เพื่อใช้วิธีนี้ เริ่มจาก Circuit ที่ใช้งาน time-evolution ที่ต้องการ และสร้าง models ที่แทนการดำเนินการเหล่านี้จาก layers ของ Circuit ที่สอดคล้องกัน สุดท้าย object `Evolver` จะถูกสร้างขึ้นซึ่งสามารถใช้สร้างปริมาณ time-evolved $M_{i,j}(t)$ และ $L_i(t)$ ได้ เริ่มต้นโดยการสร้าง object `Evolver` ที่สอดคล้องกับ approximate time-evolution ([`ApproxEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ApproxEvolverFactory)) ที่ใช้งานโดย Circuit โดยเฉพาะอย่างยิ่ง ให้ใส่ใจตัวแปร `order` เป็นพิเศษเพื่อให้ตรงกัน สังเกตว่าในการสร้าง Circuit ที่สอดคล้องกับ approximate time-evolution เราใช้ค่า placeholder สำหรับ `time = 1.0` และจำนวน Trotter steps (`reps=1`) Circuit ที่ประมาณค่าที่ถูกต้องจะถูกสร้างโดย dynamic problem solver ใน `setup_dynamic_lse`

In [24]:
# -------------------------Step 3-------------------------
estimator = Estimator(mode=backend)
estimator.options.default_shots = 30000

# Error suppression/mitigation
estimator.options.dynamical_decoupling.enable = True
estimator.options.twirling.enable_gates = True
estimator.options.twirling.enable_measure = True
estimator.options.twirling.num_randomizations = "auto"
estimator.options.twirling.strategy = "active-accum"
estimator.options.resilience.measure_mitigation = True
estimator.options.experimental.execution_path = "gen3-turbo"

estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 1.2, 1.4)
estimator.options.resilience.zne.extrapolator = "linear"

estimator.options.environment.job_tags = ["TUT_MPF"]

job_50 = estimator.run(
    [
        (circ, observable)
        for circ, observable in zip(transpiled_circuits, isa_observables)
    ]
)

> **Warning:** ตัวเลือกของ `LayerwiseEvolver` ที่กำหนดรายละเอียดของ tensor network simulation ต้องถูกเลือกอย่างระมัดระวังเพื่อหลีกเลี่ยงการตั้งค่าปัญหา optimization ที่ไม่ถูกต้อง

จากนั้นเราตั้งค่า exact evolver (เช่น [`ExactEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ExactEvolverFactory)) ซึ่งคืน object [`Evolver`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.html#qiskit_addon_mpf.backends.Evolver) ที่คำนวณ time-evolution ที่แท้จริงหรือ "reference" ในทางปฏิบัติ เราจะประมาณ exact evolution โดยใช้สูตร Suzuki–Trotter อันดับสูงกว่าหรือวิธีที่เชื่อถือได้อื่น ๆ ที่มี time step เล็ก ด้านล่างนี้ เราประมาณสถานะ exact time-evolved ด้วยสูตร Suzuki-Trotter อันดับสี่โดยใช้ time step เล็ก `dt=0.1` ซึ่งหมายความว่าจำนวน Trotter steps ที่ใช้ในเวลา $t$ คือ $k=t/dt$ เรายังระบุตัวเลือก truncation เฉพาะของ TeNPy เพื่อจำกัด maximum bond dimension ของ underlying tensor network รวมถึง singular values ต่ำสุดของ split tensor network bonds พารามิเตอร์เหล่านี้สามารถส่งผลต่อความแม่นยำของค่าคาดหวังที่คำนวณด้วย dynamic MPF coefficients ดังนั้นสิ่งสำคัญคือต้องสำรวจค่าต่าง ๆ เพื่อหาสมดุลที่เหมาะสมระหว่างเวลาคำนวณและความแม่นยำ สังเกตว่าการคำนวณ MPF coefficients ไม่ได้อาศัยค่าคาดหวังของ PF ที่ได้จากการรันบน hardware ดังนั้นจึงสามารถปรับแต่งใน post-processing ได้

In [25]:
# -------------------------Step 4-------------------------
result = job_50.result()
evs = [res.data.evs for res in result]
std = [res.data.stds for res in result]

print(evs)
print(std)

[array(-0.07916195), array(-0.04479681), array(-0.2560756), array(-0.06045848)]
[array(0.04605538), array(0.10056336), array(0.14426151), array(0.04059092)]


In [26]:
exact_mpf_std = np.sqrt(
    sum([(coeff**2) * (std**2) for coeff, std in zip(mpf_coeffs, std[:3])])
)
print(
    "Exact static MPF expectation value: ",
    evs[:3] @ mpf_coeffs,
    "+-",
    exact_mpf_std,
)
approx_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(coeffs_approx.value, std[:3])
        ]
    )
)
print(
    "Approximate static MPF expectation value: ",
    evs[:3] @ coeffs_approx.value,
    "+-",
    approx_mpf_std,
)
dynamic_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(mpf_dynamic_coeffs, std[:3])
        ]
    )
)
print(
    "Dynamic MPF expectation value: ",
    evs[:3] @ mpf_dynamic_coeffs,
    "+-",
    dynamic_mpf_std,
)

Exact static MPF expectation value:  -0.5665938395816946 +- 0.3925273058119915
Approximate static MPF expectation value:  -0.25856647611537903 +- 0.164249927266166
Dynamic MPF expectation value:  -0.12667812062949296 +- 0.06059471006973169


In [27]:
sym = {3: "^", 4: "s", 6: "p"}
for k, step in enumerate(mpf_trotter_steps):
    plt.errorbar(
        k,
        evs[k],
        yerr=std[k],
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

plt.errorbar(
    3,
    evs[-1],
    yerr=std[-1],
    alpha=0.5,
    markersize=8,
    marker="x",
    color="blue",
    label="10 Trotter steps",
)

plt.errorbar(
    4,
    evs[:3] @ mpf_coeffs,
    yerr=exact_mpf_std,
    markersize=4,
    marker="o",
    color="purple",
    label="Static MPF",
)

plt.errorbar(
    5,
    evs[:3] @ coeffs_approx.value,
    yerr=approx_mpf_std,
    markersize=4,
    marker="o",
    color="orange",
    label="Approximate static MPF",
)

plt.errorbar(
    6,
    evs[:3] @ mpf_dynamic_coeffs,
    yerr=dynamic_mpf_std,
    markersize=4,
    marker="o",
    color="pink",
    label="Dynamic MPF",
)

exact_obs = -0.24384471447172074  # Calculated via Tensor Network calculation
plt.axhline(
    y=exact_obs, linestyle="--", color="red", label="Exact time-evolution"
)

plt.title(
    f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ at time {total_time} for the different methods"
)
plt.xlabel("Method")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/64360d85-0.avif" alt="Output of the previous code cell" />

A few observations about the hardware results above:

- **Going deeper is not free on hardware.** The single-circuit baselines tell the story directly: the $k = 6$ circuit is essentially exact ($-0.256$ versus the reference $-0.244$), yet the deeper $k = 10$ baseline is *worse* ($-0.061$, off by $\sim 0.18$), not better. Once the Trotter error is already small, adding steps mostly deepens the circuit and accumulates more gate noise and decoherence. This is precisely the regime MPFs are built for: reach the accuracy of a deep circuit using only shallow constituents.

- **A small-norm MPF beats the deep single circuit.** The approximate-static MPF (capped at $\|x\|_1 \approx 2$) lands at $-0.259$, within $\sim 0.015$ of the reference and far closer than the $k = 10$ baseline. The dynamic MPF ($-0.127$) also comfortably beats that baseline. Both combine only the shallow $k_j = [3, 4, 6]$ circuits, yet recover an answer the deep single circuit could not.

- **Coefficient norm matters more than mathematical optimality.** The exact-static MPF has $\|x\|_1 = 4.66$ and is the *worst* estimator of all ($-0.567$, off by more than $0.3$): the large coefficient norm amplifies the residual gate noise, decoherence, and ZNE error on each $\langle A \rangle_{k_j}$ by roughly the same factor, overwhelming the Trotter-error cancellation it buys. Capping the norm (the approximate-static solver, $\|x\|_1 \approx 2$) removes this overwhelm and gives the best estimate &mdash; even though its coefficients no longer cancel the leading Trotter error exactly.

- **Individual shallow circuits can still be competitive.** The lone $k = 6$ constituent ($-0.256$) is itself essentially exact here &mdash; on this run it is even marginally closer than the approximate-static MPF. The catch is that you do not know in advance *which* single $k$ sits in the sweet spot of "converged but not yet noise-limited," and the safe-looking choice of simply going deeper ($k = 10$) to guarantee Trotter convergence is exactly the one that fails. The MPF gives a principled combination of shallow circuits that does not require guessing the right depth.

The practical takeaway is that on hardware, MPFs should be paired with strong error mitigation on each individual $\langle A \rangle_{k_j}$, the coefficient $L_1$-norm should be kept modest (use the approximate solver, or the dynamic MPF), and the Trotter steps $k_j$ should be chosen so that $t/k_{\min} \lesssim 1$ &mdash; here $k_{\min} = 3$ at $t = 3$ gives $t/k_{\min} = 1$, keeping the constituents inside the convergent regime where the leading-error model the static MPF relies on is valid. With those choices, the small-norm MPFs here match a converged single circuit while the naive "just go deeper" baseline does not, recovering the depth-versus-accuracy advantage shown in Ref. [\[3\]](#references). Note also that individual runs are noisy &mdash; on a different submission of the same job (or a different backend), the exact ordering can shift; the robust trends are that small-$\|x\|_1$ MPFs do well, the large-$\|x\|_1$ exact-static MPF is amplified by hardware noise, and the over-deep single circuit is noise-limited.

## Next steps
<Admonition type="tip" title="Recommendations">

If you found this work interesting, you might be interested in the following material:
- [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html) — practical guidance on selecting $k_j$ values to avoid instabilities
- [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html) — tuning the $L_1$-norm constraint and solver options for the approximate static MPF
- [`qiskit-addon-mpf` API reference](https://qiskit.github.io/qiskit-addon-mpf/) — full documentation for static, dynamic, and backend modules
</Admonition>

## References

\[1] Vazquez, A. C., Egger, D. J., Ochsner, D., & Woerner, S. Well-conditioned multi-product formulas for hardware-friendly Hamiltonian simulation. [Quantum, 7, 1067 (2023)](https://quantum-journal.org/papers/q-2023-07-25-1067/)

\[2] Zhuk, S., Robertson, N. F., & Bravyi, S. Trotter error bounds and dynamic multi-product formulas for Hamiltonian simulation. [Physical Review Research, 6(3), 033309 (2024)](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.6.033309)

\[3] Robertson, N. F., et al. Tensor network enhanced dynamic multiproduct formulas. [arXiv:2407.17405 (2024)](https://arxiv.org/abs/2407.17405)